# Documento tecnico del proyecto de gestion presupuestal

## Fundacion Uniban y sus unidades de negocio

Este cuaderno funciona como anexo tecnico del proyecto. Resume las herramientas utilizadas, la forma de acceso a Databricks, los scripts de carga, las transformaciones principales y el papel de apoyo de MCP durante el desarrollo.

**Fecha:** 20 de mayo de 2026

## 1. Herramientas utilizadas

![Arquitectura general](assets/arquitectura_bigdata.svg)

| Capa | Herramienta | Uso |
| --- | --- | --- |
| Modelo previo | Excel, Power Query, Power Pivot | Transformacion y presentacion financiera institucional |
| Desarrollo | VS Code, PowerShell, Python 3 | Edicion, ejecucion de scripts y validaciones locales |
| Lectura de fuentes | openpyxl, zipfile, xml.etree.ElementTree | Lectura de xlsx/xlsm, tablas y metadatos |
| Plataforma de datos | Databricks SQL Warehouse | Ejecucion de SQL y almacenamiento en Bronze, Silver y Gold |
| Integracion | API SQL Statements | Envio de sentencias SQL sin usar drivers pesados |
| Seguridad | Personal Access Token de Databricks | Autenticacion con cabecera Bearer |
| Consumo | app.py y report_service.py | Consulta web multiusuario |
| Soporte de ingenieria | GitHub Copilot y herramientas tipo MCP | Exploracion del repositorio, verificacion y documentacion |

**Aclaracion:** MCP se uso como apoyo de desarrollo. No forma parte del runtime productivo del sistema.

## 2. Acceso a Databricks, token y configuracion

La autenticacion real del proyecto se hace con `DATABRICKS_HOST`, `DATABRICKS_TOKEN` y `DATABRICKS_WAREHOUSE_ID`. El codigo del repositorio no depende del Databricks CLI para ejecutar cargas; la operacion principal se resolvio con Python y la API HTTP de Databricks.

Configuracion esperada en `.env` o `.env.example`:

```env
DATABRICKS_HOST=https://dbc-xxxx.cloud.databricks.com
DATABRICKS_TOKEN=
DATABRICKS_WAREHOUSE_ID=
DATABRICKS_CATALOG=presupuesto
DATABRICKS_SCHEMA=gold
DBX_QUERY_CONTRACT=gold
```

Fragmento base del cliente en `databricks_client.py`:

```python
headers = {
    "Authorization": f"Bearer {self.settings.token}",
    "Content-Type": "application/json",
}

result = self._request(
    "POST",
    "/api/2.0/sql/statements",
    {
        "statement": statement,
        "warehouse_id": self.settings.warehouse_id,
        "catalog": self.settings.catalog,
        "schema": self.settings.schema,
        "wait_timeout": wait_timeout,
        "disposition": "INLINE",
    },
)
```

**Punto de sustentacion:** el token debe mantenerse fuera de GitHub y el archivo `.env` no se publica.

## 3. Scripts principales y comandos de trabajo

Archivos tecnicos principales del proyecto:

- `config.py`: carga variables de entorno y contrato de consultas.
- `databricks_client.py`: cliente HTTP para Databricks.
- `deploy_contract.py`: despliega tablas y vistas del dominio presupuestal.
- `sync_bronze_inventory.py`: registra el inventario de archivos fuente.
- `run_full_pipeline.py`: ejecuta la carga end-to-end de presupuesto y ejecucion.
- `app.py`: expone la aplicacion web.

Comandos usados desde la carpeta del proyecto:

```powershell
python deploy_contract.py --catalog presupuesto
python sync_bronze_inventory.py --catalog presupuesto
python run_full_pipeline.py --catalog presupuesto
python -B app.py
```

Variantes utiles para pruebas controladas:

```powershell
python deploy_contract.py --catalog presupuesto --dry-run
python sync_bronze_inventory.py --catalog presupuesto --dry-run
python run_full_pipeline.py --catalog presupuesto --dry-run
python run_full_pipeline.py --catalog presupuesto --skip-bronze-raw
```

## 4. Codigo de carga desde la hoja Datos

Una mejora central del proyecto fue tomar las tablas maestras reales desde la hoja `Datos` de los libros de consolidacion. El pipeline no depende solo de heuristicas.

Fragmento representativo de `run_full_pipeline.py`:

```python
for file_path in _iter_consolidation_files(consolidation_root):
    table_refs = _read_sheet_table_refs(file_path, "Datos")
    if not table_refs:
        continue

    workbook = openpyxl.load_workbook(file_path, read_only=True, data_only=True)
    worksheet = workbook["Datos"]

    for row in _iter_table_rows(worksheet, table_refs.get("Rubros_Ingresos", "")):
        ...

    for row in _iter_table_rows(worksheet, table_refs.get("Rubros_Gastos", "")):
        ...

    for row in _iter_table_rows(worksheet, table_refs.get("HomologacionCeco", "")):
        ...

    for row in _iter_table_rows(worksheet, table_refs.get("Clasificacion_ceco", "")):
        ...
```

Tablas usadas para homologacion y relacion con presupuesto:

- `Rubros_Ingresos`
- `Rubros_Gastos`
- `HomologacionCeco`
- `Clasificacion_ceco`

## 5. Codigo de transformacion y clasificacion

La parte mas sensible estuvo en relacionar la ejecucion contable con rubros y conceptos presupuestales. El pipeline separa ingresos y gastos, y deja trazabilidad de la calidad del mapeo.

Fragmento representativo de `run_full_pipeline.py`:

```python
if line_type == "INCOME":
    income_mapping = execution_lookup_tables.income_rubrics.get((account_code, source_center_code))
    if income_mapping is not None:
        center_code = income_mapping.homologated_center_code or source_center_code
        mapped_concept = income_mapping.mapped_budget_concept or "Otros ingresos"
        rubric_current = income_mapping.rubric_current or mapped_concept
        rubric_summary = income_mapping.rubric_summary or rubric_current
        mapping_quality = "TABLE_INCOME"
else:
    homologation = _resolve_homologation(execution_lookup_tables, fiscal_year, source_center_code)
    expense_concept = _resolve_expense_concept(execution_lookup_tables, fiscal_year, center_code)
    expense_rubric, rubric_quality = _resolve_expense_rubric(execution_lookup_tables, account_code)
    fallback_concept, fallback_quality = _map_execution_concept(account_name, comment, line_type)
    ...
```

Valores tecnicos de calidad del mapeo usados en el proyecto:

- `TABLE_INCOME`
- `TABLE_EXPENSE`
- `TABLE_EXPENSE_CONCEPT`
- `TABLE_EXPENSE_ACCOUNT`
- `TABLE_EXPENSE_GROUP`
- `ACCOUNT_NAME_FALLBACK`
- `RULE_BASED`
- `DEFAULT_FALLBACK`

## 6. Transformaciones por capas

![Pipeline medallion](assets/pipeline_medallion.svg)

| Capa | Objetivo | Tablas principales |
| --- | --- | --- |
| Bronze | Trazabilidad e inventario del dato fuente | `execution_file_inventory`, `approved_budget_file_inventory`, `approved_budget_rows_raw`, `execution_rows_raw` |
| Silver | Normalizacion de negocio | `approved_budget_lines`, `execution_transactions`, `cost_center_mappings`, `financial_concept_mappings` |
| Gold | Consumo analitico | `dim_classification`, `dim_cost_center`, `dim_financial_concept`, `fact_budget_monthly`, `fact_execution_monthly`, `vw_*` |

La app consulta principalmente Gold, pero la calidad final depende de que Silver conserve una homologacion correcta entre centro de costo, rubro y concepto.

## 7. Rol de MCP y cierre tecnico

Durante el desarrollo se utilizo apoyo de GitHub Copilot con herramientas tipo MCP para explorar el workspace, leer archivos, validar notebooks y revisar cambios. Eso agilizo la ingenieria y la documentacion, pero no reemplaza el pipeline ni Databricks como plataforma de ejecucion.

Puntos clave para sustentar:

1. El proyecto migra un modelo analitico institucional desde Excel, Power Query y Power Pivot hacia una arquitectura medallion.
2. El acceso a Databricks se hace con token de acceso personal y API SQL Statements.
3. Las cargas se separan en scripts de contrato, inventario y pipeline end-to-end.
4. Las transformaciones se apoyan en tablas maestras reales de la hoja `Datos`.
5. La capa Gold deja lista la consulta para la aplicacion web y para analisis posteriores.

---

### Evidencia real de las tablas desplegadas

La siguiente captura muestra el detalle de las tablas y vistas creadas en cada capa del catalogo `presupuesto` en Databricks. Con esto puedo mostrar que la arquitectura medallion no se quedo en el diseno, sino que quedo desplegada con objetos reales en Bronze, Silver y Gold.

![Detalle Gold, Bronze y Silver](assets/databricks_medallion_tables.png)


## 8. Como sustento este proyecto en la materia de Big Data

En mi caso, yo no lo presento como un proyecto de Big Data solo por volumen. Lo sustento porque integra varias fuentes institucionales, formaliza reglas de negocio, organiza el dato por capas y deja una capa de consumo reutilizable para consulta y seguimiento.

### Lo que hice
- Integre presupuesto aprobado, ejecucion presupuestal y libros de consolidacion.
- Organice el flujo en Bronze, Silver y Gold para separar origen, transformacion y consumo.
- Formalice homologaciones y reglas del negocio para que el resultado no dependiera solo de trabajo manual en hojas de calculo.
- Deje la capa Gold lista para alimentar una aplicacion web multiusuario.

### Como lo explicaria en la sustentacion
Yo lo explicaria como una modernizacion de la analitica financiera institucional con enfoque de ingenieria de datos y practicas de Big Data. Para mi, el valor no esta solo en el numero de filas, sino en la integracion de fuentes, la trazabilidad del proceso y la posibilidad de publicar una capa analitica lista para uso.

### Alcance real de la entrega
Mi objetivo no fue quedarme en una idea de dashboard futuro. Lo que estoy desarrollando es una plataforma web multiusuario, con apoyo de ClaudeCode y Codex, para mostrar a los stakeholders el presupuesto y su ejecucion. Si despues se requiere conectar Power BI o Databricks Dashboards, la capa Gold ya lo permite, pero ese no es el centro de esta entrega.
